In [ ]:
import numpy 
import os
import scipy
import ipywidgets
from scipy.ndimage import label, center_of_mass, sum as ndi_sum
import scipp as sc
from scippneutron.conversion import graph
import matplotlib.pyplot as plt

import ipywidgets as widgets

import inspect
import plopp

import read_h5
import plot_data_ddict
import magic_graphs
import magic_scipp
import peak_find
import get_ub
import integrate_peaks
import operations_with_da

import importlib
# importlib.reload(operations_with_da)

In [ ]:
def assign_dg_to_da_coords(dg: sc.DataGroup, da: sc.DataArray, prefix:str=""):
    for s_key in dg.keys():
        da.coords[f"{prefix}_{s_key}"] = dg[s_key]

In [ ]:
f_nexus_data = r"/Users/iuriikibalin/Downloads/sim_c60/mccode.h5"
f_nexus_data = r"/Users/iuriikibalin/Repositories/magic/sim_c60_all/mccode.h5"
f_nexus_data = r"/Users/iuriikibalin/Repositories/magic/sim_c60_one/mccode.h5"
dg_magic = read_h5.read_magic_from_nexus(f_nexus_data)
dg_magic

In [ ]:
import scippnexus
dg_mcstas = scippnexus.load(f_nexus_data)

In [ ]:
dg_mcstas['entry1']['instrument']['components']['0114_monitor_at_sample']['output']['monitor_at_sample_1782224615_t']

In [ ]:
dg_sample = dg_magic['sample']
da_det_a = dg_magic['detector_a']

In [ ]:
da_laue = da_det_a.groupby('voxel_ID').sum('event')
da_laue

In [ ]:
da_det_a

In [ ]:
ind_max = numpy.argmax(da_laue.data.values)
voxel_id = da_laue.coords['voxel_ID'][ind_max]
voxel_id

In [ ]:
voxel_id

In [ ]:
da_det_a = da_det_a.transform_coords(("event_gamma",'event_nu', 'event_position_global'), graph=magic_graphs.graph_detector, rename_dims=False)

In [ ]:
assign_dg_to_da_coords(dg_magic['sample'], da_det_a, prefix="sample")
da_det_a.coords['tp_position'] = dg_magic['tp_position']
da_det_a.coords['source_position'] = dg_magic['source_position']
da_det_a.coords['delta_L'] = dg_magic['delta_L']
da_det_a.coords['delta_t'] = dg_magic['delta_t']
da_det_a = da_det_a.transform_coords(("Q_vec_rot","norm_Q", "two_theta"), graph=magic_graphs.graph_qvec)

In [ ]:
da_det_a

In [ ]:
# Given by User
cell_a = sc.scalar(14.04078, unit="angstrom")
cell_b = sc.scalar(14.04078, unit="angstrom")
cell_c = sc.scalar(14.04078, unit="angstrom")
cell_alpha = sc.scalar(90., unit="deg")
cell_beta = sc.scalar(90., unit="deg")
cell_gamma = sc.scalar(90., unit="deg")

# First estimation
euler_alpha = sc.scalar(0., unit="deg")
euler_beta = sc.scalar(0., unit="deg")
euler_gamma = sc.scalar(0., unit="deg")

In [ ]:
da_det_a.coords["cell_a"] = cell_a
da_det_a.coords["cell_b"] = cell_b
da_det_a.coords["cell_c"] = cell_c
da_det_a.coords["cell_alpha"] = cell_alpha
da_det_a.coords["cell_beta"] = cell_beta
da_det_a.coords["cell_gamma"] = cell_gamma
da_det_a.coords["euler_alpha"] = euler_alpha
da_det_a.coords["euler_beta"] = euler_beta
da_det_a.coords["euler_gamma"] = euler_gamma
magic_scipp.remove_coords_in_da(da_det_a, "h", "k", "l", "h_reduced", "k_reduced", "l_reduced", "u_matrix", "ub_matrix", "b_matrix")

da_det_a = da_det_a.transform_coords(("h","k","l","h_reduced","k_reduced","l_reduced"), graph=magic_graphs.graph_hkl)  

In [ ]:
flag = da_det_a.coords['voxel_ID'] == voxel_id


In [ ]:
da_det_a_pixel = da_det_a[flag]

In [ ]:
da_det_a.coords.keys()

In [ ]:
da_det_a_pixel.hist(k=20).plot()

In [ ]:
bin_toa = sc.linspace(dim='toa', start=0.0555, stop=0.0655, num=1001,endpoint=True, unit='s')
da_det_a_pixel.hist(toa=bin_toa).plot()

In [ ]:
x = da_det_a_pixel.hist(toa=bin_toa).coords['toa'].values

In [ ]:
y = da_det_a_pixel.hist(toa=bin_toa).data.values

In [ ]:
numpy.savetxt('x.txt',x)
numpy.savetxt('y.txt',y)

In [ ]:
da_det_a_pixel.hist(toa=bin_toa).plot()

In [ ]:

da_det_a_pixel.hist(toa=bin_toa).plot()

In [ ]:

da_det_a_pixel.hist(toa=bin_toa).plot()

In [ ]:
(bin_toa[1]-bin_toa[0]).to(unit='micros')

In [ ]:
dg_magic['cave_monitor'].hist(toa=10000).plot()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


# ------------------------------------------------------------
# 1. Reciprocal lattice from unit cell
# ------------------------------------------------------------
def reciprocal_lattice(a, b, c, alpha, beta, gamma):
    # Convert to radians
    alpha, beta, gamma = map(np.deg2rad, (alpha, beta, gamma))

    # Direct lattice metric tensor
    volume = a*b*c*np.sqrt(
        1 + 2*np.cos(alpha)*np.cos(beta)*np.cos(gamma)
        - np.cos(alpha)**2 - np.cos(beta)**2 - np.cos(gamma)**2
    )

    astar = (b*c*np.sin(alpha)) / volume
    bstar = (a*c*np.sin(beta)) / volume
    cstar = (a*b*np.sin(gamma)) / volume

    return astar, bstar, cstar


# ------------------------------------------------------------
# 2. Generate (h,k,l) grid
# ------------------------------------------------------------
def generate_hkl(hmax, kmax, lmax):
    h = np.arange(-hmax, hmax+1)
    k = np.arange(-kmax, kmax+1)
    l = np.arange(-lmax, lmax+1)
    H, K, L = np.meshgrid(h, k, l, indexing='ij')
    return np.vstack([H.ravel(), K.ravel(), L.ravel()]).T


# ------------------------------------------------------------
# 3. Compute Q vectors in lab frame using UB matrix
# ------------------------------------------------------------
def compute_Q_vectors(hkl, UB):
    return (UB @ hkl.T).T


# ------------------------------------------------------------
# 4. Ewald sphere limits
# ------------------------------------------------------------
def ewald_limits(lambda_min, lambda_max):
    kmin = 2*np.pi / lambda_max
    kmax = 2*np.pi / lambda_min
    return kmin, kmax


# ------------------------------------------------------------
# 5. Detector angular coverage (γ, ν)
# ------------------------------------------------------------
def detector_mask(Q, gamma_min, gamma_max, nu_min, nu_max):
    # Convert to spherical angles
    x, y, z = Q[:,0], Q[:,1], Q[:,2]
    r = np.linalg.norm(Q, axis=1)
    gamma = np.rad2deg(np.arctan2(y, x))     # horizontal angle
    nu    = np.rad2deg(np.arcsin(z / r))     # vertical angle

    mask = (
        (gamma >= gamma_min) & (gamma <= gamma_max) &
        (nu    >= nu_min)    & (nu    <= nu_max)
    )
    return mask


# ------------------------------------------------------------
# 6. Plotting
# ------------------------------------------------------------
def plot_reciprocal_space(Q, hkl, mask):
    fig = plt.figure(figsize=(10,8))
    ax = fig.add_subplot(111, projection='3d')

    # All points
    ax.scatter(Q[:,0], Q[:,1], Q[:,2], s=5, alpha=0.1, color='gray')

    # Points inside detector coverage
    Qdet = Q[mask]
    hkldet = hkl[mask]
    ax.scatter(Qdet[:,0], Qdet[:,1], Qdet[:,2], s=20, color='red')

    # Label visible peaks
    for q, (h,k,l) in zip(Qdet, hkldet):
        ax.text(q[0], q[1], q[2], f"({h}{k}{l})", fontsize=8)

    ax.set_xlabel("Qx (Å⁻¹)")
    ax.set_ylabel("Qy (Å⁻¹)")
    ax.set_zlabel("Qz (Å⁻¹)")
    ax.set_title("3D Reciprocal Space with Detector Coverage")
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 7. Example usage
# ------------------------------------------------------------
if __name__ == "__main__":
    # Unit cell (example)
    a, b, c = 4.5, 4.5, 6.0
    alpha, beta, gamma = 90, 90, 120

    # UB matrix (example)
    UB = np.array([
        [0.12, 0.00, 0.00],
        [0.00, 0.12, 0.00],
        [0.00, 0.00, 0.08]
    ])

    # Generate HKL grid
    hkl = generate_hkl(5, 5, 5)

    # Compute Q vectors
    Q = compute_Q_vectors(hkl, UB)

    # Wavelength range
    lambda_min = 1.5
    lambda_max = 2.5
    kmin, kmax = ewald_limits(lambda_min, lambda_max)

    # Detector angular limits (example)
    gamma_min, gamma_max = -30, 30
    nu_min, nu_max = -10, 10

    # Apply detector mask
    mask = detector_mask(Q, gamma_min, gamma_max, nu_min, nu_max)

    # Plot
    plot_reciprocal_space(Q, hkl, mask)

